# Per-Host Benchmark Metrics

In [ ]:
from provenance_explorer.registry.registry_all import WORK, CACHE_ROOT
from pathlib import Path
import pickle
import json
import pandas as pd
import numpy as np

pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:.6g}')

from provenance_explorer.analysis.provenance_capture.correctness import (
    compute_gaps_per_host,
    extract_timing_errors,
    compute_timespan_per_host,
)
from provenance_explorer.analysis.system_scale.volumetrics import (
    compute_volumetrics,
)
from provenance_explorer.analysis.system_scale.workload_variability import (
    fit_hmm_select_k,
)
from provenance_explorer.analysis.activity_realism.activity_evolution.evolution_metrics import (
    compute_activity_evolution_metrics,
)

## Data Loading

In [ ]:
path_counts = CACHE_ROOT / "events_per_host_plot"
file_counts = "events_per_host.parquet"

path_timing = CACHE_ROOT / "timestamp_disorder_plot"
file_timing = "timestamp_disorder.json"

path_object_lookup = CACHE_ROOT / "object_lookups"
file_object_lookup = lambda ds, subds: path_object_lookup / f"{ds}_{subds}_meta.pkl"

path_evolution = CACHE_ROOT / "activity_evolution_plot"
file_evolution = "activity_evolution.parquet"

def _load_pkl(path: Path):
    with open(path, 'rb') as fh:
        return pickle.load(fh)

def _load_json(path: Path):
    with open(path) as fh:
        return json.load(fh)

def _load_parquet(path: Path):
    return pd.read_parquet(path)

ds_subds = [
    ("e3", "cadets"),
    ("e3", "clearscope"),
    ("e3", "fivedirections"),
    ("e3", "theia"),
    ("e3", "trace"),
    ("e5", "cadets"),
    ("e5", "clearscope"),
    ("e5", "fivedirections"),
    ("e5", "marple"),
    ("e5", "theia"),
    ("e5", "trace"),
    ("optc", "aia_51_75"),
    ("optc", "aia_201_225"),
    ("optc", "aia_501_525"),
    ("optc", "aia_951_975"),
]

## Compute Metrics

In [ ]:
all_rows = []
bic_curves = {}

for ds, subds in ds_subds:
    print(f"Processing {ds}/{subds}...")

    event_counts = _load_parquet(path_counts / ds / subds / file_counts)
    timing_errors = _load_json(path_timing / ds / subds / file_timing)
    evolution_ledger = _load_parquet(path_evolution / ds / subds / file_evolution)

    # 1. Gaps
    df_gaps = compute_gaps_per_host(event_counts)

    # 2. Timing errors
    df_timing = extract_timing_errors(timing_errors)

    # 3. Volumetrics (events/s, infoflow/s)
    df_vol = compute_volumetrics(event_counts)

    # 4. Workload variability (GHMM + BIC)
    df_wl = fit_hmm_select_k(event_counts, k_max=15, n_restarts=5, bin_size_minutes=15)

    # BIC curves for plotting
    for _, row in df_wl.iterrows():
        bic_curves[(ds, subds, row["host_id"])] = row["bic_curve"]

    # Drop bic_curve column before merge (not tabular)
    df_wl_tab = df_wl.drop(columns=["bic_curve"])

    # 5. Timespan (first-to-last bin per host)
    df_timespan = compute_timespan_per_host(event_counts)

    # 6. Activity evolution (unique normalised cmdlines + saturation AUC)
    df_evo = compute_activity_evolution_metrics(evolution_ledger, ds, subds)

    # Merge all per-host frames
    df = df_gaps.merge(df_timing, on="host_id", how="outer")
    df = df.merge(df_vol, on="host_id", how="outer")
    df = df.merge(df_wl_tab, on="host_id", how="outer")
    df = df.merge(df_timespan, on="host_id", how="outer")
    df = df.merge(df_evo, on="host_id", how="outer")

    # Add dataset identifiers
    df.insert(0, "dataset", ds)
    df.insert(1, "sub_dataset", subds)

    all_rows.append(df)
    print(f"{len(df)} host(s) done.")

metrics_table = pd.concat(all_rows, ignore_index=True)
print(f"\nTotal: {len(metrics_table)} rows")

## Full Results Table

In [ ]:
display_cols = [
    "dataset", "sub_dataset", "host_id",
    # Timespan
    "timespan_days",
    # Provenance Capture
    "gap_pct",
    "timing_error_rate", "timing_mean_error_s", "timing_std_error_s", "timing_max_error_s",
    # System Scale - Volumetrics
    "avg_events_per_s", "std_events_per_s",
    "avg_infoflow_per_s", "std_infoflow_per_s",
    # System Scale - Workload Variability
    "best_k",
    # Activity Realism - Activity Evolution
    "n_unique_normalized_cmdlines", "saturation_auc_unit",
]

metrics_table[display_cols]

In [ ]:
metrics_table[display_cols].to_excel("display_table.xlsx")
metrics_table.to_csv("full_table.csv", index=False)

## BIC Curves sanity check

In [ ]:
import matplotlib.pyplot as plt
from provenance_explorer.analysis.system_scale.workload_variability import (
    plot_bic_curve,
)

# Pick first host from each dataset family for illustration
examples = {}
for (ds, subds, host), curve in bic_curves.items():
    if ds not in examples and curve:
        examples[ds] = (ds, subds, host, curve)

fig, axes = plt.subplots(1, len(examples), figsize=(6 * len(examples), 3.5))
if len(examples) == 1:
    axes = [axes]

for ax, (ds, (ds_, subds, host, curve)) in zip(axes, examples.items()):
    plot_bic_curve(curve, title=f"{ds_}/{subds}\n{host[:20]}...", ax=ax)

plt.tight_layout()
plt.show()

## State Sequence Sanity Check

In [ ]:
from provenance_explorer.analysis.system_scale.workload_variability import (
    plot_state_sequence,
)

fig, axes = plt.subplots(len(examples), 1, figsize=(14, 3.5 * len(examples)))
if len(examples) == 1:
    axes = [axes]

for ax, (ds, (ds_, subds, host, curve)) in zip(axes, examples.items()):
    event_counts = _load_parquet(path_counts / ds_ / subds / file_counts)
    best_k = min(curve, key=lambda x: x[1])[0]
    plot_state_sequence(event_counts, host, best_k, ax=ax)

plt.tight_layout()
plt.show()

## Aggregation Exploration

In [ ]:
# For multi-host sub-datasets, show coefficient of variation for key metrics
key_metrics = [
    "gap_pct", "avg_events_per_s", "avg_infoflow_per_s", "best_k",
    "timespan_days", "n_unique_normalized_cmdlines", "saturation_auc_unit",
]

multi_host = (
    metrics_table
    .groupby(["dataset", "sub_dataset"])
    .filter(lambda g: len(g) > 1)
)

if len(multi_host) > 0:
    agg = (
        multi_host
        .groupby(["dataset", "sub_dataset"])[key_metrics]
        .agg(["mean", "std", "count"])
    )
    print("Multi-host sub-datasets — mean and std of key metrics:")
    display(agg)
else:
    print("All sub-datasets have a single host.")